## LoRA implementation

In [1]:
import torch
import torch.nn as nn
from GPTModules import TransformerBlock,LayerNorm
class GPTModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["dim"])
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["dim"])
        self.dropout=nn.Dropout(cfg["drop_rate"])
        self.trf_blocks=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )
        self.layer_norm=LayerNorm(cfg["dim"])
        self.out_head=nn.Linear(
            cfg["dim"],cfg["vocab_size"],bias=False
        )
    def forward(self,input):
        batch_size,context_len=input.shape
        tok_emb=self.tok_emb(input)
        pos_emb=self.pos_emb(torch.arange(context_len,device=input.device))
        x=tok_emb+pos_emb
        x=self.dropout(x)
        x=self.trf_blocks(x)
        x=self.layer_norm(x)
        x=self.out_head(x)
        return x


In [2]:
config={
        'vocab_size': 50257,
    'context_length': 1024,
    'drop_rate': 0.0,
    'qkv_bias': True,
    'dim': 1024,
    'num_layers': 24,
    'num_heads': 16
    }

In [3]:
model=GPTModel(config)
model

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (dropout): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiheadAttention(
        (W_q): Linear(in_features=1024, out_features=1024, bias=True)
        (W_k): Linear(in_features=1024, out_features=1024, bias=True)
        (W_v): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiheadAttention(
        (W_q): Linear(in_features=1024, ou

In [4]:
# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)

Trainable parameters: 406286336


In [8]:
class Lora(nn.Module):
    def __init__(self,inp_dim:int,out_dim:int,rank:int):
        super().__init__()
        self.A=nn.Parameter(torch.rand(inp_dim,rank))
        self.B=nn.Parameter(torch.rand(rank,out_dim))
    def forward(self,x):
        result=x @ self.A @ self.B
        return result



In [15]:
torch.manual_seed(123)
x=torch.tensor([[1.0,2.0,3.0],[3.0,4.0,6.0]])
lora=Lora(x.shape[1],4,rank=1)
y=lora(x)
y

tensor([[1.4351, 0.1542, 1.8060, 0.2847],
        [3.0741, 0.3303, 3.8687, 0.6098]], grad_fn=<MmBackward0>)

In [7]:
x=torch.rand(3,2)
x

tensor([[0.0518, 0.4442],
        [0.4758, 0.0471],
        [0.4311, 0.1100]])